# 웨이퍼맵 결함 분류 — 1회차

**AI를 활용한 반도체 공정 분석 · 3회 집중 과정**

---

| 구간 | 언제 |
|---|---|
| PART 1~3 | 수업에서 함께 봅니다 |
| PART 4 | 과제 ① 틀린 이미지 확인 |
| PART 5 | AI 에게 설명시켜 보기 |
| PART 6 | 과제 ② 30장 직접 나누기 |

### 규칙

- **코드는 읽지 말고 실행만** 하세요. 셀을 위에서부터 순서대로 누르면 됩니다.
- 여러분이 바꾸는 것은 **몇 개의 숫자와 폴더 이름**뿐입니다.
- 오늘 볼 것은 코드가 아니라 **결과를 어떻게 읽을 것인가**입니다.

> 2회차 실습은 **다른 파일**로 진행합니다. 이 파일은 오늘 것만 담고 있습니다.

---
# PART 0 — 준비

아래 두 셀만 실행하면 됩니다.

In [ ]:
import os, glob, random, zipfile, shutil, re
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

try:
    from google.colab import files
except Exception:
    files = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| 사용 장치:', device)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'gdown'],
               capture_output=True, text=True)
import gdown

ZIP_URL = 'https://drive.google.com/file/d/1eZzS5HgefKFhNjwFBdEFYdIT1C8dF40l/view?usp=sharing'

DATA_DIR = '/content/wafer'
shutil.rmtree(DATA_DIR, ignore_errors=True)
os.makedirs(DATA_DIR, exist_ok=True)

JUNK = ('__MACOSX', '.DS_Store', 'Thumbs.db')

def clean_junk(base):
    """압축 프로그램이 끼워 넣는 파일을 지웁니다.
    맥에서 압축하면 ._사진.jpg 같은 가짜 이미지가 섞여 들어와 학습 중 에러가 납니다."""
    for p in sorted(glob.glob(os.path.join(base, '**', '*'), recursive=True), key=len, reverse=True):
        name = os.path.basename(p)
        if name in JUNK or name.startswith('._'):
            try:
                shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
            except OSError:
                pass

m = (re.search(r'/d/([A-Za-z0-9_-]{20,})', ZIP_URL)
     or re.search(r'[?&]id=([A-Za-z0-9_-]{20,})', ZIP_URL))
fid = m.group(1) if m else ZIP_URL.strip()
dst = '/content/wafer_set.zip'

ok = None
for attempt in (lambda: gdown.download(id=fid, output=dst, quiet=False),
                lambda: gdown.download(f'https://drive.google.com/uc?id={fid}', dst, quiet=False)):
    try:
        ok = attempt()
        if ok: break
    except TypeError:
        continue
    except Exception as e:
        print('  시도 실패:', type(e).__name__)

# 여기서 멈춰야 합니다. 그냥 넘어가면 한참 뒤에 엉뚱한 에러로 나타납니다.
if not (ok and zipfile.is_zipfile(dst)):
    raise RuntimeError('데이터를 받지 못했습니다.\n'
                       '  1) ZIP_URL 이 zip 파일 링크가 맞는지\n'
                       '  2) 공유 설정이 "링크가 있는 모든 사용자" 인지 확인하세요.')

with zipfile.ZipFile(dst) as z:
    z.extractall(DATA_DIR)
clean_junk(DATA_DIR)

print('\n준비 완료')
for d in sorted(glob.glob(os.path.join(DATA_DIR, '*'))):
    if os.path.isdir(d):
        n = len(glob.glob(os.path.join(d, '**', '*.*'), recursive=True))
        print(f'  {os.path.basename(d):16} {n}장')

---
# PART 1 — 웨이퍼맵은 어떻게 생겼나

In [ ]:
CLASS_NAMES = ['Donut', 'Scratch']

# Donut / Scratch 폴더가 어디에 있든 찾아냅니다.
def find_class_root(base):
    for h in sorted(glob.glob(os.path.join(base, '**', CLASS_NAMES[0]), recursive=True)):
        if not os.path.isdir(h) or '__MACOSX' in h.replace(os.sep, '/').split('/'):
            continue
        root = os.path.dirname(h)
        if all(os.path.isdir(os.path.join(root, c)) for c in CLASS_NAMES):
            return root
    return None

TRAIN_DIR = find_class_root(DATA_DIR)

if TRAIN_DIR is None:
    print('[확인 필요] Donut / Scratch 폴더를 찾지 못했습니다. 받은 내용:')
    for f in sorted(glob.glob(os.path.join(DATA_DIR, '**', '*'), recursive=True))[:20]:
        print('   ', os.path.relpath(f, DATA_DIR))
    raise FileNotFoundError('Donut / Scratch 폴더가 없습니다. 위 목록을 확인하세요.')

print('학습 폴더:', TRAIN_DIR)
for c in CLASS_NAMES:
    fs = glob.glob(os.path.join(TRAIN_DIR, c, '*.*'))
    print(f'  {c:10} {len(fs)}장')

# 한 장을 크게 봅니다 (원본 색 그대로)
sample = sorted(glob.glob(os.path.join(TRAIN_DIR, CLASS_NAMES[0], '*.*')))[0]
im = Image.open(sample).convert('RGB')
print(f'\n예시: {os.path.basename(sample)}  크기 {im.size}  (가로 x 세로 die 수)')
plt.figure(figsize=(4, 4))
plt.imshow(im, interpolation='nearest')
plt.axis('off'); plt.show()

**먼저 눈에 익히세요.**

윗줄이 Donut, 아랫줄이 Scratch 입니다. 마젠타색이 불량 die 입니다.

- Donut 은 가운데를 비우고 **고리 모양**으로 퍼집니다.
- Scratch 는 **한 방향으로 길게** 이어집니다.

지금 이 차이가 눈에 안 들어오면, 뒤에서 모델이 틀린 것을 봐도 왜 틀렸는지 알 수 없습니다.

In [ ]:
# 두 종류가 어떻게 다른지 나란히 놓고 봅니다
N = 5
fig, axes = plt.subplots(2, N, figsize=(13, 5.8))
for r, c in enumerate(CLASS_NAMES):
    fs = sorted(glob.glob(os.path.join(TRAIN_DIR, c, '*.*')))[:N]
    for k in range(N):
        ax = axes[r, k]; ax.axis('off')
        if k < len(fs):
            ax.imshow(Image.open(fs[k]).convert('RGB'), interpolation='nearest')
            ax.set_title(c, fontsize=11)
plt.tight_layout(); plt.show()

**폴더 이름이 곧 정답표입니다.**

```
Wafermap/            ← 압축을 풀면 나오는 폴더 (이름은 배포본마다 다를 수 있습니다)
├─ Donut/     ← 이 안에 있으면 Donut
└─ Scratch/   ← 이 안에 있으면 Scratch
```

폴더 이름 자체는 중요하지 않습니다. 위 셀이 `Donut` / `Scratch` 가 나란히 있는 곳을
알아서 찾습니다.

누군가 이 폴더에 파일을 넣어두었고, 모델은 그 판단을 그대로 배웁니다.
**뒤에서 여러분이 직접 그 “누군가”가 됩니다.**

---
# PART 2 — CNN으로 분류해 보기

아래 세 셀은 **실행만** 하세요. 하는 일은 이렇습니다.

```
  이미지를 64×64로 맞춘다  →  8:2로 나눈다  →  학습한다  →  채점한다
```

In [ ]:
IMG_SIZE, BATCH_SIZE = 64, 16

# 웨이퍼맵은 배경/정상/불량 세 값만 있는 격자입니다.
# 크기를 바꿀 때 NEAREST를 써야 칸의 의미가 유지됩니다.
base_tf = [transforms.Grayscale(1),
           transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=InterpolationMode.NEAREST),
           transforms.ToTensor(),
           transforms.Normalize([0.5], [0.5])]
tf = transforms.Compose(base_tf)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, num_classes)
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

print('준비 완료')

In [ ]:
class WaferFolder(datasets.ImageFolder):
    """Donut / Scratch 두 폴더만 클래스로 씁니다.
    (mixed 같은 폴더가 옆에 있어도 3번째 클래스로 끼어들지 않게 합니다.)"""
    def find_classes(self, directory):
        missing = [c for c in CLASS_NAMES if not os.path.isdir(os.path.join(directory, c))]
        if missing:
            raise FileNotFoundError(f'{directory} 안에 폴더가 없습니다: {missing}')
        return list(CLASS_NAMES), {c: i for i, c in enumerate(CLASS_NAMES)}


def run_cnn(folder, seed=42, epochs=15, quiet=False, eval_folder=None, exclude=()):
    """폴더를 읽어 CNN을 학습하고 결과를 돌려줍니다.

    eval_folder 를 주면 학습은 folder 로 하고 채점은 그 폴더로 합니다.
    exclude 는 채점에서 뺄 파일 이름 목록입니다.
    """
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)

    ds = WaferFolder(folder, transform=tf)
    ys = [y for _, y in ds.samples]
    counts = {c: ys.count(i) for i, c in enumerate(ds.classes)}
    if min(counts.values()) < 2:
        raise ValueError(f'각 폴더에 최소 2장은 있어야 합니다. 지금: {counts}')

    if eval_folder is None:
        tr, va = train_test_split(range(len(ds)), test_size=0.2,
                                  stratify=ys, random_state=seed)
        train_ds, val_ds = Subset(ds, tr), Subset(ds, va)
        val_src, val_idx = ds, va
    else:
        eds = WaferFolder(eval_folder, transform=tf)
        skip = set(exclude)
        keep = [i for i, (p, _) in enumerate(eds.samples) if os.path.basename(p) not in skip]
        train_ds, val_ds = ds, Subset(eds, keep)
        val_src, val_idx = eds, keep

    tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    vl = DataLoader(val_ds, batch_size=BATCH_SIZE)

    model = SimpleCNN(len(ds.classes)).to(device)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for ep in range(1, epochs + 1):
        model.train()
        for x, y in tl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); loss = crit(model(x), y)
            loss.backward(); opt.step()
        if not quiet and ep % 5 == 0:
            print(f'  {ep:2d} epoch 진행')

    model.eval(); preds, tgts = [], []
    with torch.no_grad():
        for x, y in vl:
            preds += model(x.to(device)).argmax(1).cpu().tolist()
            tgts += y.tolist()

    return {
        'acc': accuracy_score(tgts, preds),
        'f1': f1_score(tgts, preds, average='macro'),
        'cm': confusion_matrix(tgts, preds, labels=list(range(len(ds.classes)))),
        'classes': ds.classes,
        'val_idx': val_idx, 'preds': preds, 'targets': tgts, 'ds': val_src,
        'n_train': len(train_ds), 'n_val': len(val_ds),
    }

def show_result(r, title=''):
    print(f"{title}  정확도 {r['acc']:.3f}   F1 {r['f1']:.3f}"
          f"   (학습 {r['n_train']}장 / 채점 {r['n_val']}장)")
    # 한쪽으로만 답한 경우를 알려줍니다
    if len(set(r['preds'])) == 1:
        only = r['classes'][r['preds'][0]]
        print(f"    [주의] 모델이 전부 '{only}' 라고만 답했습니다.")
        print("           정확도 숫자는 나왔지만 실제로는 아무것도 구분하지 못한 상태입니다.")
    cm, cls = r['cm'], r['classes']
    plt.figure(figsize=(3.6, 3.2))
    plt.imshow(cm, cmap='Blues')
    plt.xticks(range(len(cls)), cls); plt.yticks(range(len(cls)), cls)
    plt.xlabel('Predicted'); plt.ylabel('True (folder)')
    for i in range(len(cls)):
        for j in range(len(cls)):
            plt.text(j, i, cm[i, j], ha='center', va='center')
    plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()

print('준비 완료')

In [ ]:
r1 = run_cnn(TRAIN_DIR, seed=42)
show_result(r1, '기본 학습')

### 표를 읽는 법

```
                모델의 답
               Donut  Scratch
정답  Donut  [  맞음   틀림  ]
    Scratch  [  틀림   맞음  ]
```

대각선이 맞힌 것, 나머지가 틀린 것입니다.

**정확도 숫자 하나만 보지 마세요.** 어느 쪽으로 틀렸는지가 더 중요합니다.

---
# PART 3 — 같은 코드인데 결과가 다릅니다

아무것도 바꾸지 않고, **숫자 하나(seed)만** 바꿔서 세 번 더 돌려봅니다.

데이터도 같고 코드도 같습니다. 결과가 같을까요?

In [ ]:
results = {}
for s in [42, 7, 123, 2024]:
    r = run_cnn(TRAIN_DIR, seed=s, quiet=True)
    results[s] = r
    flag = '   ← 한쪽으로만 답함' if len(set(r['preds'])) == 1 else ''
    print(f'seed {s:>4}   정확도 {r["acc"]:.3f}   F1 {r["f1"]:.3f}{flag}')

accs = [r['acc'] for r in results.values()]
print(f'\n최고 {max(accs):.3f}  /  최저 {min(accs):.3f}  /  차이 {max(accs)-min(accs):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
for ax, (s, r) in zip(axes, results.items()):
    k = len(r['classes'])
    ax.imshow(r['cm'], cmap='Blues')
    ax.set_xticks(range(k)); ax.set_xticklabels(r['classes'], fontsize=8)
    ax.set_yticks(range(k)); ax.set_yticklabels(r['classes'], fontsize=8)
    for i in range(k):
        for j in range(k):
            ax.text(j, i, r['cm'][i, j], ha='center', va='center')
    ax.set_title(f"seed {s}  ({r['acc']:.2f})", fontsize=10)
plt.tight_layout(); plt.show()

### 왜 다를까요

seed는 **데이터를 어떻게 나눌지, 학습을 어디서 시작할지**를 정하는 숫자입니다.
그것만 바꿔도 결과가 달라집니다.

> **여기서 기억할 것**
>
> 정확도 0.92 라는 숫자는 “이 모델의 실력”이 아니라
> **“이번에 이렇게 나왔다”** 는 뜻입니다.
>
> 숫자 하나를 보고 판단하면 안 되는 이유입니다.

### 한쪽으로만 답한 경우가 있었나요

위 결과에 **“한쪽으로만 답함”** 표시가 붙은 것이 있다면 눈여겨보세요.

그 모델은 모든 이미지에 같은 답을 냈습니다. 아무것도 구분하지 못한 것입니다.
그런데도 정확도는 0.5 근처로 나옵니다. 둘 중 하나는 맞으니까요.

> **정확도가 나왔다고 모델이 작동한 것은 아닙니다.**
> 표의 대각선만 보지 말고, 한 줄이 통째로 비어 있지 않은지 확인하세요.

---
# PART 4 — 1회차 과제

### 1) 틀린 이미지를 직접 보기

모델이 틀린 것들을 눈으로 확인합니다. **사람이 봐도 헷갈리는지** 판단해 보세요.

In [ ]:
r = results[42]
wrong = [(i, p, t) for i, (p, t) in enumerate(zip(r['preds'], r['targets'])) if p != t]
print(f'틀린 개수: {len(wrong)} / {len(r["preds"])}')

if wrong:
    n = min(len(wrong), 6)
    fig, axes = plt.subplots(1, n, figsize=(2.4*n, 3))
    if n == 1: axes = [axes]
    for ax, (i, p, t) in zip(axes, wrong[:n]):
        path = r['ds'].samples[r['val_idx'][i]][0]
        ax.imshow(Image.open(path).convert('RGB'), interpolation='nearest')
        ax.set_title(f"true: {r['classes'][t]}\npred: {r['classes'][p]}", fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('이번에는 다 맞혔습니다. 다른 seed로 바꿔 다시 돌려보세요.')

### 2) 제출할 내용

**① seed를 바꿔 4번 돌린 결과를 적으세요.** (정확도 4개)

**② 최고와 최저의 차이는 얼마였나요?**
이 정도 차이면 “이 모델은 정확도 ○○이다”라고 말할 수 있을까요?

**③ 틀린 이미지 중 하나를 골라 두 줄로 쓰세요.**
사람이 봐도 헷갈릴 만한 이미지였나요, 아니면 명백한데 모델이 틀린 건가요?

> 정답을 맞히는 과제가 아닙니다. **결과를 어떻게 읽었는지**를 봅니다.

---
# PART 5 — AI 는 이 이미지를 뭐라고 설명할까

**순서를 지키세요.**

```
  ① 이미지를 크게 본다  →  ② 내 판정을 적는다  →  ③ AI 준비  →  ④ AI 에게 묻는다
```

AI 답을 먼저 보면 거기에 끌려갑니다. **내 판단을 먼저 글로 박아두는 것**이 핵심입니다.

> 키는 Colab 왼쪽 **열쇠 아이콘 🔑** 에 `GEMINI_API_KEY` 로 등록해 두세요.
> **노트북 액세스** 토글을 켜야 합니다.

In [ ]:
# ① 이미지를 고르고 눈으로 봅니다 (AI 키가 없어도 돌아갑니다)
def show_big(path, scale=8):
    """작은 웨이퍼맵을 키워서 봅니다. NEAREST 를 쓰는 이유는 앞에서와 같습니다."""
    im = Image.open(path).convert('RGB')
    print(os.path.basename(path), '|', im.size, '(가로 x 세로 die 수)')
    plt.figure(figsize=(5, 5))
    plt.imshow(im, interpolation='nearest'); plt.axis('off'); plt.show()
    return im

# CNN 이 틀린 이미지
_r = results[42]
WRONG = [_r['ds'].samples[_r['val_idx'][i]][0]
         for i, (p, t) in enumerate(zip(_r['preds'], _r['targets'])) if p != t]
print('틀린 이미지', len(WRONG), '장')

PICK = 0                      # ← 0, 1, 2 ... 바꾸면 다른 이미지를 봅니다
TARGET = WRONG[PICK]

_ = show_big(TARGET)

In [ ]:
# ② 위 이미지를 보고 내 판정을 적습니다. 적은 뒤 이 셀을 실행하세요.
MY_CALL   = 'Donut'                 # ← 'Donut' 또는 'Scratch'
MY_REASON = '가운데가 비어 보여서'      # ← 왜 그렇게 봤는지 한 줄

assert MY_CALL in CLASS_NAMES, f'{CLASS_NAMES} 중 하나로 적으세요'
print('적었습니다 →', MY_CALL, '|', MY_REASON)

In [ ]:
# AI 준비 - 한 번만 실행하면 됩니다
PROVIDER = 'gemini'        # ← 'gemini' 또는 'openai' 로 바꿀 수 있습니다
API_KEY  = ''              # ← 보통 비워둡니다. 보안 비밀을 못 쓸 때만 여기에 붙여넣으세요.

import base64, io as _io

def _get_key(name):
    """Colab 왼쪽 열쇠 아이콘(보안 비밀)에서 키를 읽습니다.

    못 읽으면 입력창을 띄우지 않고 바로 멈춥니다.
    입력창을 띄우면 그것을 못 본 사람은 셀이 영원히 도는 것처럼 보입니다."""
    if API_KEY.strip():
        return API_KEY.strip()
    try:
        from google.colab import userdata
        k = userdata.get(name)
        if k:
            return k
        why = '값이 비어 있습니다'
    except Exception as e:
        why = type(e).__name__
    raise RuntimeError(
        f'{name} 를 읽지 못했습니다 ({why}).\n'
        '  1. 왼쪽 열쇠 아이콘 🔑 클릭\n'
        f'  2. 이름을 정확히  {name}  으로 새 보안 비밀 추가\n'
        '  3. [노트북 액세스] 토글 켜기   ← 이걸 빼먹는 경우가 가장 많습니다\n'
        '  4. 이 셀을 다시 실행\n'
        '  (그래도 안 되면 이 셀 맨 위 API_KEY 에 키를 직접 붙여넣으세요)')

def _big(im, scale=8):
    """웨이퍼맵은 아주 작습니다. 키워서 보내야 AI 가 볼 것이 생깁니다.
    NEAREST 를 쓰는 이유는 앞에서와 같습니다 - 없던 중간색이 생기면 안 됩니다."""
    im = im.convert('RGB')
    return im.resize((im.width * scale, im.height * scale), Image.NEAREST)

AI_MODEL = None

if PROVIDER == 'gemini':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'google-genai'],
                   capture_output=True, text=True)
    from google import genai
    from google.genai import types as _gt
    os.environ['GEMINI_API_KEY'] = _get_key('GEMINI_API_KEY')
    _client = genai.Client()
    _avail = [m.name.split('/')[-1] for m in _client.models.list()]
    for _n in ['gemini-3.6-flash', 'gemini-3.5-flash', 'gemini-2.5-flash']:
        if _n in _avail:
            AI_MODEL = _n; break
    if AI_MODEL is None:
        AI_MODEL = next((m for m in _avail if 'flash' in m), None)

    def ask_ai(path, prompt):
        r = _client.models.generate_content(
            model=AI_MODEL,
            contents=[prompt, _big(Image.open(path))],
            config=_gt.GenerateContentConfig(
                http_options=_gt.HttpOptions(timeout=60_000)))
        return r.text

else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'openai'],
                   capture_output=True, text=True)
    from openai import OpenAI
    _client = OpenAI(api_key=_get_key('OPENAI_API_KEY'))

    def ask_ai(path, prompt):
        buf = _io.BytesIO(); _big(Image.open(path)).save(buf, format='PNG')
        url = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
        last = None
        for name in ['gpt-5', 'gpt-4.1', 'gpt-4o']:
            try:
                r = _client.chat.completions.create(model=name, messages=[{
                    'role': 'user',
                    'content': [{'type': 'text', 'text': prompt},
                                {'type': 'image_url', 'image_url': {'url': url}}]}])
                globals()['AI_MODEL'] = name
                return r.choices[0].message.content
            except Exception as e:
                last = e
        raise RuntimeError(f'쓸 수 있는 모델을 찾지 못했습니다: {type(last).__name__}')

print('AI 준비 완료 -', PROVIDER, AI_MODEL or '(첫 호출 때 정해집니다)')

In [ ]:
# ④ AI 에게 던질 질문입니다. ★ 따옴표 안쪽을 바꿔가며 실험해 보세요.
PROMPT = ('이 이미지는 반도체 웨이퍼 맵입니다. '
          '시안색이 정상 die, 마젠타색이 불량 die, 흰색은 웨이퍼 바깥입니다.\n'
          '먼저 이미지에서 관찰되는 사실만 3가지를 쓰고, '
          '그 다음 줄에 Donut 인지 Scratch 인지 한 단어로 쓰세요.\n'
          '판단이 어려우면 "애매함"이라고 답해도 됩니다.')

import time as _time

print('[내가 던진 질문]')
print(PROMPT)
print()
print('AI 에게 보내는 중...', end=' ')
_t0 = _time.time()
_answer = ask_ai(TARGET, PROMPT)
print(f'{_time.time() - _t0:.0f}초 걸렸습니다')

# 답의 뒤쪽부터 훑어 결론 한 단어를 찾습니다
_lines = [l.strip() for l in _answer.splitlines() if l.strip()]
_ai_call = None
for _l in reversed(_lines):
    for _c in list(CLASS_NAMES) + ['애매함']:
        if _c.lower() in _l.lower():
            _ai_call = _c
            break
    if _ai_call:
        break

# 결론 줄은 관찰에서 빼고 보여줍니다
_obs = _lines[:-1] if (_ai_call and _lines and _ai_call.lower() in _lines[-1].lower()) else _lines

print()
print('=' * 60)
print('[AI 가 말한 관찰]')
for _l in _obs:
    print('  ', _l)

print()
print('AI 판정 :', _ai_call or '(한 단어로 답하지 않았습니다)')
print('내 판정 :', MY_CALL, '  |  ', MY_REASON)
print()

if _ai_call is None:
    print('-> AI 가 결론을 한 단어로 주지 않았습니다. 프롬프트를 조여 보세요.')
elif _ai_call == '애매함':
    print('-> AI 는 판단을 미뤘습니다. 여러분은 하나를 골라야 했습니다.')
    print('   그 차이가 무엇을 뜻하는지 생각해 보세요.')
elif _ai_call == MY_CALL:
    print('-> 같습니다. 그렇다고 둘 다 맞는다는 뜻은 아닙니다.')
    print('   위 관찰 3가지가 실제로 이미지에 있는지 확인하세요.')
else:
    print('-> 갈렸습니다. 누구 근거가 더 이미지에 붙어 있습니까?')

### 답을 읽을 때 볼 것

- AI 가 말한 **관찰 3가지가 실제로 이미지에 있습니까?** 없는 것을 지어내지 않았나요.
- 결론이 내 판정과 **같은가요, 다른가요.** 다르다면 누구 근거가 더 이미지에 붙어 있나요.
- AI 는 **"모르겠다"고 말하지 않습니다.** 확신 없이도 하나를 고릅니다.
  문장이 유창한 것과 맞는 것은 아무 상관이 없습니다.

> `PICK` 이나 파일 이름을 바꿔가며 두세 장 더 돌려보세요.

---
# PART 6 — 과제 ② 30장을 직접 나누기

**다음 주 수업은 이 결과에서 시작합니다.**

지금까지는 남이 만들어둔 폴더로 학습했습니다.
이번에는 **여러분이 직접 나눈 폴더**를 만들어 옵니다.

```
  ① 아래 셀을 실행해 이미지 30장을 내려받는다
        ↓
  ② 내 컴퓨터에서 Donut / Scratch 로 나눈다
        ↓
  ③ my_labels 폴더째 압축해서 다음 주에 가져온다
```

In [ ]:
# mixed 폴더 찾기 -> 내 컴퓨터로 내려받기 (한 번만)
def find_mixed(base):
    for d in sorted(glob.glob(os.path.join(base, '**', 'mixed'), recursive=True)):
        if os.path.isdir(d) and glob.glob(os.path.join(d, '*.*')):
            return d
    return None

N_MIXED = 30
MIXED_KEY = []          # 아래 대체 경로로 만들 때만 채워집니다 (강사용)
MIXED_DIR = find_mixed(DATA_DIR)

if MIXED_DIR is not None:
    print('배포된 mixed 폴더를 사용합니다:', MIXED_DIR)
else:
    # 배포 zip 에 mixed 가 없을 때만 쓰는 임시 대체입니다.
    # 원칙은 라벨이 없는 원본에서 뽑은 mixed 를 배포 zip 에 넣는 것입니다.
    MIXED_DIR = os.path.join(DATA_DIR, 'mixed')
    os.makedirs(MIXED_DIR, exist_ok=True)
    rng = random.Random(0)
    picked = []
    per = N_MIXED // len(CLASS_NAMES)
    for c in CLASS_NAMES:
        fs = sorted(glob.glob(os.path.join(TRAIN_DIR, c, '*.*')))
        picked += [(f, c) for f in rng.sample(fs, min(per, len(fs)))]
    rng.shuffle(picked)
    for i, (src, c) in enumerate(picked, 1):
        name = f'wafer_{i:03d}{os.path.splitext(src)[1]}'
        shutil.copy(src, os.path.join(MIXED_DIR, name))
        MIXED_KEY.append((name, c, os.path.basename(src)))
    with open('/content/mixed_answer.csv', 'w', encoding='utf-8') as fp:
        fp.write('file,original_label,original_file\n')
        for row in MIXED_KEY:
            fp.write(','.join(row) + '\n')
    print(f'[대체] 배포 zip 에 mixed 가 없어 학습 폴더에서 {len(MIXED_KEY)}장을 뽑았습니다.')
    print('  (강사용 정답표: /content/mixed_answer.csv - 학생에게는 배포하지 마세요)')

MIXED_SRC = [k[2] for k in MIXED_KEY]   # 채점에서 뺄 원본 파일명 (배포 mixed 면 비어 있음)

shutil.make_archive('/content/mixed', 'zip', MIXED_DIR)
print('분류할 이미지:', len(glob.glob(os.path.join(MIXED_DIR, '*.*'))), '장')
if files is not None:
    files.download('/content/mixed.zip')
else:
    print('파일 위치: /content/mixed.zip')

## 나누는 방법

받은 `mixed.zip` 을 풀고, 옆에 이런 폴더를 만들어 옮기세요.

```
my_labels/
├─ Donut/
└─ Scratch/
```

### 규칙 — 반드시 둘 중 하나를 고르세요

**"애매함" 폴더는 만들지 마세요.** 헷갈려도 하나로 정해야 합니다.

불편할 겁니다. 그런데 실제 데이터 라벨링이 그렇게 이루어집니다.
누군가 헷갈리면서 하나를 골랐고, 그 판단이 지금 우리가 쓰는 "정답"이 되었습니다.

---

### 많이 헷갈릴 겁니다 — 그게 정상입니다

이 30장은 **모델이 가장 판단하기 어려워한 것들만** 모아 놓은 것입니다.
쉽게 갈리지 않습니다.

그래도 둘 중 하나를 고르세요. 오래 붙들지 말고, 헷갈린 것은 따로 적어 두었다가
수업에서 이야기하면 됩니다.

## 제출할 것

**①** 다 나눈 `my_labels` 폴더를 **압축**해 `my_labels.zip` 으로 만들어 오세요.
다음 주 실습 파일에서 그대로 올리게 됩니다.

**②** PART 4의 제출 내용(정확도 4개 · 차이 · 틀린 이미지 한 장 설명)도 함께 정리해 오세요.

**③** 나누면서 **가장 오래 망설인 이미지 3장의 파일 이름**을 적어 오세요.
왜 망설였는지 한 줄씩 붙이면 좋습니다.

---
## 사용한 데이터

본 실습의 웨이퍼맵 이미지는 **공개 데이터셋**에서 가져온 것입니다.
포트폴리오나 자소서에 쓸 때는 공개 데이터셋임을 반드시 밝히십시오.
“실제 공정 데이터를 분석했다”고 쓰면 면접에서 문제가 됩니다.